# Accessing Claude with the API

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

Notebook này đi qua các khái niệm cốt lõi của phần *"Accessing Claude with the API"*, từ cơ bản đến nâng cao một chút. Bạn đọc giải thích (ô markdown) rồi chạy từng ô code để thấy kết quả.


## 0. Cài đặt & API Key

- **API Key** là "mật khẩu" để Anthropic biết ai đang gọi và tính tiền đúng tài khoản. Lấy tại [console.anthropic.com](https://console.anthropic.com).
- **Không hardcode key** trong code. Đặt qua biến môi trường:

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```


In [ ]:
# Cài SDK (chỉ chạy 1 lần)
!pip install anthropic

## 1. Tạo Client

`client` là "cánh cổng" kết nối tới Anthropic. Tạo một lần, dùng nhiều lần. Mặc định SDK tự đọc key từ biến môi trường `ANTHROPIC_API_KEY`.


In [ ]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"   # model mạnh nhất hiện tại

## 2. Gọi API đầu tiên (Basic Request)

Một request cần **3 thành phần bắt buộc**:

| Thành phần | Ý nghĩa |
|---|---|
| `model` | Chọn "bộ não" nào trả lời |
| `max_tokens` | Độ dài tối đa của câu trả lời |
| `messages` | Lịch sử hội thoại (danh sách tin nhắn) |

Mỗi tin nhắn có `role` (`"user"` hoặc `"assistant"`) và `content`.


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Thủ đô của Việt Nam là gì?"}
    ],
)

# response.content là 1 DANH SÁCH block. Lấy text ở block đầu tiên:
print(response.content[0].text)

## 3. Hiểu cấu trúc Response

Đối tượng `response` chứa nhiều thông tin hữu ích, không chỉ text:
- `id`, `model`, `role`
- `stop_reason`: lý do dừng
- `usage`: số token đã dùng (để tính chi phí 💰)


In [ ]:
print("id:          ", response.id)
print("model:       ", response.model)
print("role:        ", response.role)
print("stop_reason: ", response.stop_reason)
print("usage:       ", response.usage)

# Cách an toàn để lấy text: duyệt block và check type
for block in response.content:
    if block.type == "text":
        print("Câu trả lời:", block.text)

## 4. System Prompt

System prompt là chỉ dẫn cấp cao, định hình **vai trò/giọng văn** của Claude cho TOÀN BỘ cuộc hội thoại. Nó nằm ở tham số `system=` riêng, **không** nằm trong `messages`.


In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="Bạn là một giáo viên tiểu học. Hãy giải thích thật đơn giản, "
           "dùng ví dụ gần gũi với trẻ em.",
    messages=[
        {"role": "user", "content": "Giải thích trí tuệ nhân tạo là gì?"}
    ],
)
print(response.content[0].text)

## 5. Các tham số quan trọng

- **`model`**: `claude-opus-4-8` (mạnh nhất), `claude-sonnet-4-6` (cân bằng), `claude-haiku-4-5` (nhanh & rẻ).
- **`max_tokens`**: số token tối đa cho câu trả lời. 1 token ≈ 0.75 từ tiếng Anh. Đặt quá thấp → câu trả lời bị cắt (`stop_reason="max_tokens"`).

> ⚠️ Trên model mới (Opus 4.6+) **KHÔNG còn dùng `temperature`/`top_p`** như các khóa cũ. Muốn điều chỉnh độ "suy nghĩ" dùng `output_config={"effort": "low|medium|high|max"}`.


In [ ]:
# Ví dụ điều chỉnh độ "nỗ lực" suy nghĩ của model
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    output_config={"effort": "high"},
    messages=[{"role": "user", "content": "Giải thích vì sao bầu trời màu xanh."}],
)
print(response.content[0].text)

## 6. Hội thoại nhiều lượt (Multi-turn)

API là **STATELESS**: nó KHÔNG tự nhớ lượt trước. Muốn Claude nhớ, bạn phải **gửi lại toàn bộ lịch sử** mỗi lần gọi. Các vai trò phải xen kẽ: user → assistant → user → ...


In [ ]:
messages = [
    {"role": "user", "content": "Tên tôi là An."},
]

# Lượt 1
res1 = client.messages.create(model=MODEL, max_tokens=1024, messages=messages)
print("Claude:", res1.content[0].text)

# Thêm câu trả lời của Claude vào lịch sử
messages.append({"role": "assistant", "content": res1.content[0].text})

# Lượt 2 -- Claude vẫn nhớ "An" vì ta gửi lại lịch sử
messages.append({"role": "user", "content": "Tên tôi là gì?"})
res2 = client.messages.create(model=MODEL, max_tokens=1024, messages=messages)
print("Claude:", res2.content[0].text)

## 7. Stop Reason (Vì sao Claude dừng?)

`response.stop_reason` cho biết lý do dừng:

| Giá trị | Ý nghĩa |
|---|---|
| `end_turn` | Trả lời xong tự nhiên (bình thường) |
| `max_tokens` | Đụng giới hạn → câu trả lời bị cắt |
| `stop_sequence` | Gặp chuỗi dừng tùy chỉnh |
| `tool_use` | Claude muốn gọi công cụ (bài sau) |
| `refusal` | Claude từ chối vì lý do an toàn |


In [ ]:
# Ví dụ: đặt max_tokens rất nhỏ để thấy stop_reason="max_tokens"
res = client.messages.create(
    model=MODEL,
    max_tokens=5,
    messages=[{"role": "user", "content": "Kể một câu chuyện dài về biển."}],
)
print("stop_reason:", res.stop_reason)

## 8. Streaming (Hiện câu trả lời theo từng chữ)

Thay vì chờ toàn bộ câu trả lời, streaming trả về từng phần nhỏ ngay khi model sinh ra → trải nghiệm mượt, giống hiệu ứng gõ chữ.


In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "Viết một bài thơ ngắn về mùa thu."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    print()

---
## ✅ Tóm tắt

Bạn đã học: tạo client, gửi request, đọc response, system prompt, multi-turn (stateless), stop reason, streaming.

**Bài tiếp theo:** Prompt Evaluation (xem notebook `prompt_evaluation.ipynb`).
